# Jeff on the Apple Neural Engine — quick start

Download the three demo builds, compile them for this Mac's ANE, start `system1-ane-serve`, and make one decision each for
the base model (routing), Snake and Tetris.

Run from the repo with the host venv kernel (`requirements-inference.txt`). Set `COREAI_PYTHON` to your Core AI SDK
interpreter before starting Jupyter. Builds go to `$JEFF_MODELS_DIR` (default `~/Models/jeff-ane`); set
`JEFF_TETRIS_BUILD` etc. to use local builds instead. Set `SYSTEM1_ANE_URL` to reuse a running server (the old `JEFF_URL` still works).

In [ ]:
import os, sys, time, statistics
from pathlib import Path
sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "jeff_nb.py").exists() else Path.cwd() / "notebooks"))
import jeff_nb as J
print("repo:", J.REPO)
print("models dir:", J.models_dir())
print("COREAI_PYTHON:", os.environ.get("COREAI_PYTHON", "<not set>"))

## 1. Download (skipped for builds already present)

In [ ]:
builds = {name: J.ensure(name) for name in ("base", "snake", "tetris")}
for name, path in builds.items():
    print(f"{name:7s} {path}  ready={J.is_build(path)}")

## 2. Compile for this Mac (once; fast when cached)

In [ ]:
if not J.reuse_url():
    for name, path in builds.items():
        t = time.time(); J.compile_build(path); print(f"{name}: compiled in {time.time() - t:.1f}s")

## 3. Start the server

In [ ]:
proc, url = J.start_server(builds["base"], {"snake": builds["snake"], "tetris": builds["tetris"]})
h = J.health(url)
print(url, h["status"], h["backend"], "adapters:", h["adapters"])

## 4. One decision per model

In [ ]:
from jeff_snake import INSTRUCTIONS as SNAKE_INSTRUCTIONS, OPTIONS as SNAKE_OPTIONS, opening_states, render_state
from jeff_tetris import generate_tetris_rows

route = {"model": "jeff-latest",
         "state": "Hi, I was charged twice for my order last week and want my money back.",
         "questions": {"route": {"type": "choice", "instructions": "Route this support message.",
                                 "criteria": {"refunds": "Refunds and billing", "shipping": "Shipping and delivery",
                                              "technical": "Technical problem", "other": "Anything else"}}}}
snake, food = opening_states(1, seed=0)[0]
snake_body = {"model": "jeff-latest", "adapter": "snake", "state": render_state(snake, food),
              "questions": {"move": {"type": "choice", "instructions": SNAKE_INSTRUCTIONS, "criteria": dict(SNAKE_OPTIONS)}}}
row = generate_tetris_rows(1, seed=0)[0]
opts = row["options"] if isinstance(row["options"], dict) else {f"o{i+1}": o for i, o in enumerate(row["options"])}
tetris_body = {"model": "jeff-latest", "adapter": "tetris", "state": row["state"],
               "questions": {"place": {"type": "choice", "instructions": row["instructions"], "criteria": opts}}}

for label, body in (("base/route", route), ("snake", snake_body), ("tetris", tetris_body)):
    out = J.decide(url, body)
    (q, ans), = out["answers"].items()
    print(f"{label:11s} choice={ans['choice']!r:12s} total={out['timings']['total_ms']:.1f} ms")

## 5. Stop the server

In [ ]:
J.stop_server(proc)

## Next steps

Training for new tasks happens off the ANE (PyTorch on a Mac with MPS, or a GPU such as Colab). The resulting adapter and choice head are then merged, converted to Core AI, compiled and served on the Apple Neural Engine on the Mac (see the README, "Build your own"). Colab and other non-Apple machines cannot compile or serve on the ANE.

## Credits

- **Base model:** Jeff v1.3, a fine-tune of Qwen3.5-0.8B by its author, [mstrasser/jeff-base](https://huggingface.co/mstrasser/jeff-base) (Apache-2.0).
- **Backbone:** Qwen3.5-0.8B by the Qwen Team, Alibaba Cloud ([Qwen/Qwen3.5-0.8B](https://huggingface.co/Qwen/Qwen3.5-0.8B), Apache-2.0).
- **Snake adapter:** a rank-16 LoRA plus choice head trained for the base model on 2026-10-07 and merged before conversion; the build shipped here comes from a local Apple M5 Max run (PyTorch MPS, `forge.py jeff-train-lora --task snake`). A separate Unsloth Snake training on a Colab A100 (also 2026-10-07) exists as an experiment; it is not what this notebook serves.
- **Tetris adapter:** LoRA plus choice head trained locally on an Apple M5 Max with PyTorch MPS, using this repo's hand-rolled LoRA (no Unsloth), in the bounded overnight rounds of 2026-10-08 (round 02 shipped). A Colab A100 continuation on 2026-10-07 failed and was discarded.
- **Training data:** generated by this repo (El-Tetris oracle for Tetris, a search oracle for Snake); no third-party dataset.